In [1]:

import numpy as np
import matplotlib.pyplot as plt

# Set environment variables
import os


os.environ["SCAL_TYPE"] = "complex"
os.environ["PRECISION"] = "single"
os.environ["MY_NUMBA_TARGET"] = "numba"

# Add cle_fun to PYTHON_PATH
import sys
sys.path.append("../../clonscal")

In [2]:
import numpy as np
from simulation.config import Config
from simulation.cl_simulation import ComplexLangevinSimulation
from src.obs_kernels import (
    n_moment_kernel, 
    dse_n_moment_kernel,
    abs_drift
)
from tqdm import tqdm
from simulation.gpu_handler import GPU_handler
from src.numba_target import use_cuda
import src.scal as scal
from src.utils import (
    update_histogram_complex, 
    update_histogram_real,
    gaussian_modified_density_drift_kernel, 
    noise_kernel_rotated,
    calculate_stats_complex
)

from src.numba_target import my_act_parallel_loop
from numba import cuda

Using single precision
Using C^1
Using Numba


define a configuration with desired parameters. see simulation.config.py for defaults. Based on the config, define a simulation object

In [3]:
config = Config(dims = [1], sigma=1, interaction=1, trajs = 2, steps = int(1e5), dt = 1e-4)
sim = ComplexLangevinSimulation(config)

Register observables into the object. This defines ObservableTracker instances

In [4]:
sim.register_observable('1_moment', obs_kernel=n_moment_kernel, const_param={"order" : 1},  langevin_history=True, thermal_time=5, auto_corr=1.0)
sim.register_observable('2_moment', obs_kernel=n_moment_kernel, const_param={"order" : 2},  langevin_history=True, thermal_time=5, auto_corr=1.0)

These can not be addressed via

In [5]:
sim.trackers

{'1_moment': <simulation.observables.ObservableTracker at 0x120bed720>,
 '2_moment': <simulation.observables.ObservableTracker at 0x117b4fd60>}

The trackers are independent from each other. The observable carries information about all the trajectories, see shape of the arrays

In [6]:
sim.trackers["1_moment"].__dict__

{'obs_name': '1_moment',
 'shape': (2,),
 'obs_kernel': CPUDispatcher(<function n_moment_kernel at 0x1208e4dc0>),
 'langevin_history': True,
 'const_param': {'order': 1},
 'init_history_size': 100000,
 'thermal_time': 5,
 'auto_corr': 1.0,
 'order': 1,
 '_sim_instance': <simulation.cl_simulation.ComplexLangevinSimulation at 0x117b4fb20>,
 'equilibrated_trajs': array([False, False]),
 'meas_time': array([-1., -1.], dtype=float32),
 'result': array([0.+0.j, 0.+0.j], dtype=complex64),
 'history_result': array([[[0.+0.j],
         [0.+0.j],
         [0.+0.j],
         ...,
         [0.+0.j],
         [0.+0.j],
         [0.+0.j]],
 
        [[0.+0.j],
         [0.+0.j],
         [0.+0.j],
         ...,
         [0.+0.j],
         [0.+0.j],
         [0.+0.j]]], dtype=complex64),
 'history_meas_times': array([[[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],
 
        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.]

Once a simulation steps is performed

In [7]:
sim.step()

the global degree of freedom sim.phi is updated according to the langevin equation. It stores the values for all the trajectories. 

In [8]:
sim.phi0

array([-0.02738328+0.j,  0.01789672+0.j], dtype=complex64)

After a step, we have to decide whether observables are to be calculated, depending on tracker.thermal_time and tracker.auto_corr.  
Every trajectory is associated with a boolean that allows or blocks calculation. These are set to False in the beginning

In [9]:
sim.trackers["1_moment"].equilibrated_trajs

array([False, False])

We could call tracker.compute(), but all trajectories will be ignored. Their result is unchanged (zero)

In [10]:
sim.trackers["1_moment"].compute()
print(sim.trackers["1_moment"].result)
print(sim.trackers["1_moment"].counter)

[0.+0.j 0.+0.j]
[0 0]


only after a certain number of steps will the trajs be allowed to be observed. To (un)block the trajs, we call tracker.mark_equilibrated_trajs

In [11]:
for _ in range(int(1e5)):
    sim.step()
    for name, tr in sim.trackers.items():
        tr.mark_equilibrated_trajs()

In [12]:
sim.trackers["1_moment"].equilibrated_trajs

array([ True,  True])

After computation, the result is stored in a buffer and the rolling stats are updated.

In [13]:
sim.trackers["1_moment"].compute()
sim.trackers["1_moment"].mark_equilibrated_trajs()
sim.trackers["1_moment"].equilibrated_trajs
print(f"Result buffer: {sim.trackers['1_moment'].result}")
print(f"Counter: {sim.trackers['1_moment'].counter}")
print(f"Rolling mean: {sim.trackers['1_moment'].rolling_mean}")
print(f"Rolling sq mean: {sim.trackers['1_moment'].rolling_sqr_mean_real}")
print()

Result buffer: [0.40605822+0.j 0.39123407+0.j]
Counter: [1 1]
Rolling mean: [0.40605822+0.j 0.39123407+0.j]
Rolling sq mean: [0.16488329 0.1530641 ]



The full loop looks someting like this

In [14]:
for _ in range(int(1e5)):
    sim.step()
    for name, tr in sim.trackers.items():
        tr.mark_equilibrated_trajs()
        tr.compute()

print(f"Result buffer: {sim.trackers['1_moment'].result}")
print(f"Counter: {sim.trackers['1_moment'].counter}")
print(f"Rolling mean: {sim.trackers['1_moment'].rolling_mean}")
print(f"Rolling sq mean: {sim.trackers['1_moment'].rolling_sqr_mean_real}")
print()

Result buffer: [-0.46602854+0.j -1.4637948 +0.j]
Counter: [10 10]
Rolling mean: [-3.6951525 +0.j  0.26762617+0.j]
Rolling sq mean: [7.0568004 5.5083423]



To combine the rolling stats over different observables, call sim.finish()

In [15]:
sim.finish()
sim.trackers["1_moment"].rolling_mean

(-3.4275265+0j)

from here stats can be calculated using utils.calculate_stats_complex

In [16]:
from src.utils import calculate_stats_complex
rolling_mean = sim.trackers["1_moment"].rolling_mean
rolling_sqr_mean_real = sim.trackers["1_moment"].rolling_sqr_mean_real
rolling_sqr_mean_imag = sim.trackers["1_moment"].rolling_sqr_mean_imag
counter = sim.trackers["1_moment"].counter
mean, sem_real, sem_imag = calculate_stats_complex(rolling_mean, rolling_sqr_mean_real, rolling_sqr_mean_imag, counter)

print(f"First moment: {np.round(mean, 5)} pm {np.round(sem_real, 5)}")

First moment: (-0.17138+0j) pm 0.17304


The code is optimized for parallel execution of multiple trajectories

In [26]:
config = Config(dims = [1], sigma=1, interaction=1, trajs = int(1e4), steps = int(1e5), dt = 1e-4)
sim = ComplexLangevinSimulation(config)

sim.register_observable('1_moment', obs_kernel=n_moment_kernel, const_param={"order" : 1},  langevin_history=True, thermal_time=2, auto_corr=1.0)
sim.register_observable('2_moment', obs_kernel=n_moment_kernel, const_param={"order" : 2},  langevin_history=True, thermal_time=2, auto_corr=1.0)

for _ in tqdm(range(sim.steps)):
    sim.step()
    for name, tr in sim.trackers.items():
        tr.mark_equilibrated_trajs()
        tr.compute()
sim.finish()

100%|██████████| 100000/100000 [00:26<00:00, 3764.78it/s]


In [18]:
# scipy routines
import numpy as np
from scipy.integrate import quad
from numba import jit

def n_moment(x, order):
    return x**order

# define rho and R
@jit
def action(x, sigma, lamb): return sigma/2*x**2+lamb/4*x**4

@jit
def rho(x, sigma, lamb): return np.exp(-action(x, sigma, lamb))

# define partition sums
def z_rho(sigma, lamb): 
    real_part = quad(lambda x: np.real(rho(x, sigma, lamb)), -np.inf, np.inf)[0]
    imag_part = quad(lambda x: np.imag(rho(x, sigma, lamb)), -np.inf, np.inf)[0]
    return real_part+1j*imag_part


# define exp val of n_moment wrt rho
@np.vectorize
def n_moment_rho(order, sigma, lamb):
    real_part = quad(lambda x: np.real(rho(x, sigma, lamb)*n_moment(x, order)), 
                     -np.inf, np.inf)[0]
    imag_part = quad(lambda x: np.imag(rho(x, sigma, lamb)*n_moment(x, order)), 
                     -np.inf, np.inf)[0]
    out = real_part + 1j*imag_part
    return out / z_rho(sigma, lamb)

In [27]:
rolling_mean = sim.trackers["1_moment"].rolling_mean
rolling_sqr_mean_real = sim.trackers["1_moment"].rolling_sqr_mean_real
rolling_sqr_mean_imag = sim.trackers["1_moment"].rolling_sqr_mean_imag
counter = sim.trackers["1_moment"].counter
mean, sem_real, sem_imag = calculate_stats_complex(rolling_mean, rolling_sqr_mean_real, rolling_sqr_mean_imag, counter)

print(f"First moment: [{np.round(mean-sem_real, 5)}; {np.round(mean+sem_real, 5)}]")
print(f"scipy: {n_moment_rho(1, sim.sigma, sim.interaction)}")

First moment: [(-0.00779+0j); (-0.00297+0j)]
scipy: 0j


In [28]:
rolling_mean = sim.trackers["2_moment"].rolling_mean
rolling_sqr_mean_real = sim.trackers["2_moment"].rolling_sqr_mean_real
rolling_sqr_mean_imag = sim.trackers["2_moment"].rolling_sqr_mean_imag
counter = sim.trackers["2_moment"].counter
mean, sem_real, sem_imag = calculate_stats_complex(rolling_mean, rolling_sqr_mean_real, rolling_sqr_mean_imag, counter)

print(f"Second moment: [{np.round(mean-sem_real, 5)}; {np.round(mean+sem_real, 5)}]")
print(f"scipy: {n_moment_rho(2, sim.sigma, sim.interaction)}")

Second moment: [(0.46649+0j); (0.47043+0j)]
scipy: (0.4679199169736833+0j)
